# 10 Data Cleaning — Detect, Fix, Verify

This notebook is a practical, executable lesson. Each step follows **Context → Source Problem → Detect → Fix → Verify → Business Result → Exercise**.

## 1. Create a deliberately dirty sales file

Real source data contains problems. We create controlled examples so every cleaning rule can be demonstrated.

In [ ]:
import pandas as pd
df=pd.DataFrame({
 'transaction_id':['T1','T2','T2','T4'],
 'sale_date':['15-02-2026','03/04/2026','03/04/2026','2026-05-04'],
 'customer_name':[' Vikas ','VIKAS','VIKAS','Anita '],
 'quantity':['2','4','4','abc'],
 'unit_price':['100','120','120','-50']
})
df.to_csv('data/dirty_sales_examples.csv',index=False)
df

## 2. Detect duplicates

Exact duplicates and duplicate business keys are different problems.

In [ ]:
dup_exact=df.duplicated(keep=False)
print(df[dup_exact])
print('duplicate transaction ids:',df[df.duplicated('transaction_id',keep=False)])

## 3. Standardize text

Whitespace/case differences cause false duplicates and failed joins.

In [ ]:
df['customer_name_clean']=df['customer_name'].str.strip().str.title()
df[['customer_name','customer_name_clean']]

## 4. Validate numeric values

Detect non-numeric and impossible values before converting.

In [ ]:
df['quantity_num']=pd.to_numeric(df['quantity'],errors='coerce')
df['unit_price_num']=pd.to_numeric(df['unit_price'],errors='coerce')
print(df[df['quantity_num'].isna() | (df['quantity_num']<=0)])
print(df[df['unit_price_num'].isna() | (df['unit_price_num']<=0)])

## 5. Date cleaning — DD-MM-YYYY vs YYYY-MM-DD

Never assume every source uses the same format. A date can be parsed incorrectly while still looking valid. For ambiguous values such as 03/04/2026, use the source contract or quarantine the record.

In [ ]:
from datetime import datetime
raw=df['sale_date']
parsed=[]
for value in raw:
    value=str(value).strip()
    result=None
    for fmt in ['%Y-%m-%d','%d-%m-%Y','%d/%m/%Y']:
        try:
            result=datetime.strptime(value,fmt); break
        except ValueError: pass
    parsed.append(result)
df['sale_date_parsed']=pd.to_datetime(parsed)
df['sale_date_std']=df['sale_date_parsed'].dt.strftime('%Y-%m-%d')
df[['sale_date','sale_date_std']]

## 6. Fix and quarantine

If a value is invalid or ambiguous, do not silently invent a date. Route it to a reject/quarantine file.

In [ ]:
rejects=df[df['quantity_num'].isna() | (df['unit_price_num']<=0)].copy()
clean=df.drop(index=rejects.index).copy()
clean['quantity']=clean['quantity_num'].astype(int)
clean['unit_price']=clean['unit_price_num']
clean.to_csv('data/clean_sales.csv',index=False)
rejects.to_csv('data/rejected_sales.csv',index=False)
print('clean:',len(clean),'rejects:',len(rejects))

## 7. Verify cleaning

Cleaning is incomplete until rules are rechecked.

In [ ]:
assert clean['quantity'].gt(0).all()
assert clean['unit_price'].gt(0).all()
assert clean['transaction_id'].is_unique
print('all cleaning checks passed')

## Exercises

1. Run every code cell and inspect the output.
2. Modify at least one input value to create a data problem.
3. Detect the problem with code.
4. Fix it without hiding the original record.
5. Verify the result and explain the business impact.